# Лабораторна робота №6

## Тема: LU -розклад. Ітераційні методи уточнення розв’язку СЛА

**Виконав:** Шипка Роман **Група:** ФЕІ-13

In [7]:
import numpy as np

# 1. Генерація матриці A (100x100) та вектора B
n = 100
A = np.random.uniform(1, 100, (n, n))

# Задаємо точний розв'язок x_true (всі елементи = 2.5) [cite: 39]
x_true = np.full(n, 2.5)

# Обчислюємо вектор вільних членів B [cite: 41]
B = A @ x_true

# Запис у текстові файли [cite: 38, 42]
np.savetxt('matrix_A.txt', A)
np.savetxt('vector_B.txt', B)

print(f"Матриця A та вектор B згенеровані та збережені. Розмірність: {n}x{n}")

Матриця A та вектор B згенеровані та збережені. Розмірність: 100x100


In [2]:
def get_lu_decomposition(A):
    n = len(A)
    L = np.zeros((n, n))
    U = np.eye(n)  # Діагональні елементи U = 1 [cite: 10]

    for k in range(n):
        # Обчислення елементів k-го стовпця L [cite: 12]
        for i in range(k, n):
            L[i, k] = A[i, k] - sum(L[i, j] * U[j, k] for j in range(k))
        
        # Обчислення елементів k-го рядка U [cite: 12]
        for i in range(k + 1, n):
            if L[k, k] == 0:
                raise ValueError("Ділення на нуль: матриця вироджена або потребує вибору головного елемента.")
            U[k, i] = (A[k, i] - sum(L[k, j] * U[j, i] for j in range(k))) / L[k, k]
            
    return L, U

def solve_lu(L, U, B):
    n = len(L)
    # 1. Розв'язок LZ = B (пряма підстановка) [cite: 16, 17]
    z = np.zeros(n)
    for k in range(n):
        z[k] = (B[k] - sum(L[k, j] * z[j] for j in range(k))) / L[k, k]
    
    # 2. Розв'язок UX = Z (зворотна підстановка) [cite: 16, 17]
    x = np.zeros(n)
    for k in range(n - 1, -1, -1):
        x[k] = z[k] - sum(U[k, j] * x[j] for j in range(k + 1, n))
        
    return x

In [3]:
# Зчитування даних (імітація функцій зчитування) [cite: 43]
A_loaded = np.loadtxt('matrix_A.txt')
B_loaded = np.loadtxt('vector_B.txt')

# Знаходження LU-розкладу [cite: 43]
L, U = get_lu_decomposition(A_loaded)
np.savetxt('LU_decomposition.txt', np.hstack((L, U))) # Збереження результату [cite: 43]

# Початковий розв'язок [cite: 44]
x_0 = solve_lu(L, U, B_loaded)

# Оцінка точності (нев'язка) [cite: 46]
def get_norm(vec):
    return np.max(np.abs(vec)) # Обчислення норми [cite: 43]

initial_error = get_norm(A_loaded @ x_0 - B_loaded)
print(f"Точність (max відхилення) початкового розв'язку: {initial_error:.2e}")

Точність (max відхилення) початкового розв'язку: 4.18e-10


In [5]:
# 5. Ітераційне уточнення [cite: 47]
eps0 = 1e-14
x_refined = x_0.copy()
max_iterations = 20
iteration = 0

print(f"{'Ітерація':<10} | {'Норма нев''язки':<20}")
print("-" * 35)

for i in range(max_iterations):
    # Обчислюємо вектор нев'язки R [cite: 23]
    R = B_loaded - A_loaded @ x_refined
    
    norm_R = get_norm(R)
    print(f"{i+1:<10} | {norm_R:<20.2e}")
    
    # Перевірка умови зупинки [cite: 31, 32]
    if norm_R <= eps0:
        iteration = i + 1
        break
        
    # Розв'язуємо A * delta_x = R, використовуючи вже готовий LU-розклад [cite: 29, 36]
    delta_x = solve_lu(L, U, R)
    
    # Уточнюємо розв'язок [cite: 30]
    x_refined = x_refined + delta_x
    iteration = i + 1

print("-" * 35)
print(f"Кількість ітерацій для досягнення точності {eps0}: {iteration}")
final_error = get_norm(x_refined - x_true)
print(f"Кінцева похибка порівняно з x_true (2.5): {final_error:.2e}")

Ітерація   | Норма невязки       
-----------------------------------
1          | 4.18e-10            
2          | 7.28e-12            
3          | 7.28e-12            
4          | 7.28e-12            
5          | 9.09e-12            
6          | 7.28e-12            
7          | 7.28e-12            
8          | 7.28e-12            
9          | 7.28e-12            
10         | 9.09e-12            
11         | 7.28e-12            
12         | 9.09e-12            
13         | 9.09e-12            
14         | 5.46e-12            
15         | 7.28e-12            
16         | 9.09e-12            
17         | 9.09e-12            
18         | 7.28e-12            
19         | 5.46e-12            
20         | 7.28e-12            
-----------------------------------
Кількість ітерацій для досягнення точності 1e-14: 20
Кінцева похибка порівняно з x_true (2.5): 1.83e-12
